# LeetCode 989 — Add to Array-Form of Integer bằng Python

**Link đề:** https://leetcode.com/problems/add-to-array-form-of-integer/
**Độ khó:** Easy · **Chủ đề:** Array, Math

```python
class Solution:
    def addToArrayForm(self, num: list[int], k: int) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu **dạng mảng** (array-form) của số nguyên và cách chuyển qua lại số ↔ mảng chữ số.
2. Thành thạo `% 10` (lấy chữ số cuối), `// 10` (bỏ chữ số cuối), `divmod`.
3. Cài đặt **phép cộng tay có nhớ** (carry) — kỹ thuật dùng lại ở Add Binary, Add Strings, Add Two Numbers, Plus One.
4. Biết mẹo hay: **coi cả `k` là "số nhớ"** để code cực gọn.
5. Trace, chứng minh, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

**Dạng mảng** của số nguyên là list các chữ số từ trái sang phải. Ví dụ `1321` → `[1, 3, 2, 1]`.

Cho `num` (dạng mảng của một số) và số nguyên `k`, trả về **dạng mảng** của `num + k`.

| num | k | Phép tính | Kết quả |
|---|:---:|---|---|
| `[1,2,0,0]` | 34 | 1200 + 34 = 1234 | `[1,2,3,4]` |
| `[2,7,4]` | 181 | 274 + 181 = 455 | `[4,5,5]` |
| `[2,1,5]` | 806 | 215 + 806 = 1021 | `[1,0,2,1]` (dài thêm 1 chữ số!) |

**Constraints:**
- `1 <= len(num) <= 10^4` → số có thể có **10 000 chữ số**!
- `0 <= num[i] <= 9`, không có số 0 ở đầu (trừ chính số 0).
- `1 <= k <= 10^4`.

## 2. Nền tảng Python

### 2.1. `% 10`, `// 10` và `divmod`

- `x % 10` → chữ số **cuối** của `x`.
- `x // 10` → `x` sau khi **bỏ** chữ số cuối (chia lấy phần nguyên).
- `divmod(x, 10)` trả về **cả hai** cùng lúc: `(x // 10, x % 10)`.

In [1]:
x = 1021
print(x % 10, x // 10)
print(divmod(x, 10))
q, r = divmod(17, 10)
print("17 -> nhớ", q, "viết", r)

1 102
(102, 1)
17 -> nhớ 1 viết 7


### 2.2. Số → mảng chữ số, và ngược lại

- Mảng → số: duyệt từ trái, `value = value * 10 + d`.
- Số → mảng: lấy `% 10` liên tục (được chữ số **ngược**), rồi đảo lại. Hoặc dùng chuỗi: `[int(c) for c in str(x)]`.

In [2]:
def to_int(digits):
    value = 0
    for d in digits:
        value = value * 10 + d
    return value

def to_digits(x):
    if x == 0:
        return [0]
    out = []
    while x > 0:
        out.append(x % 10)
        x //= 10
    out.reverse()
    return out

print(to_int([1, 2, 0, 0]))
print(to_digits(1234))
print([int(c) for c in str(1021)])

1200
[1, 2, 3, 4]
[1, 0, 2, 1]


### 2.3. `reversed`, `list.reverse()` và chỉ số âm

- `out.reverse()` đảo **tại chỗ**, trả về `None`.
- `reversed(lst)` trả về một **iterator** đi ngược, không sửa list.
- `lst[::-1]` tạo **list mới** đảo ngược.
- `num[-1]` là chữ số **cuối** (hàng đơn vị) — nơi phép cộng bắt đầu.

In [3]:
num = [2, 7, 4]
print(list(reversed(num)), num[::-1], "| num[-1] =", num[-1])

[4, 7, 2] [4, 7, 2] | num[-1] = 4


## 3. Cách trực giác (baseline) — đổi ra số, cộng, đổi lại

Python có số nguyên **không giới hạn độ lớn** (big integer), nên cách này **chạy được** trong Python.

In [4]:
def add_via_int(num: list[int], k: int) -> list[int]:
    value = int("".join(map(str, num)))     # [1,2,0,0] -> "1200" -> 1200
    return [int(c) for c in str(value + k)]

print(add_via_int([1, 2, 0, 0], 34))
print(add_via_int([2, 7, 4], 181))
print(add_via_int([2, 1, 5], 806))

[1, 2, 3, 4]
[4, 5, 5]
[1, 0, 2, 1]


Giải thích `"".join(map(str, num))`:
- `map(str, num)` áp `str` lên từng phần tử → `"1", "2", "0", "0"`.
- `"".join(...)` nối chúng lại bằng chuỗi rỗng → `"1200"`.

In [5]:
print(list(map(str, [1, 2, 0, 0])))
print("".join(["1", "2", "0", "0"]), "|", "-".join(["a", "b"]))

['1', '2', '0', '0']
1200 | a-b


## 4. Nút thắt

- Ở **Java/C++**, số 10 000 chữ số **tràn** mọi kiểu số nguyên → cách đổi ra số không dùng được. Phỏng vấn muốn thấy bạn cộng **từng chữ số**.
- Ngay cả trong Python, đổi số khổng lồ ↔ chuỗi không miễn phí: chuyển đổi `int ↔ str` với số rất lớn tốn thời gian **siêu tuyến tính**, và Python 3.11+ còn **giới hạn** mặc định 4300 chữ số cho `int(str)`! (`num` có thể tới 10⁴ chữ số.)

In [6]:
import sys
limit = getattr(sys, "get_int_max_str_digits", lambda: 0)()
print("Giới hạn chữ số khi đổi str<->int:", limit if limit else "không giới hạn (Python cũ)")
try:
    add_via_int([9] * 5000, 1)
    print("Chạy được với 5000 chữ số")
except ValueError as e:
    print("Lỗi:", str(e)[:80], "...")

Giới hạn chữ số khi đổi str<->int: 4300
Lỗi: Exceeds the limit (4300 digits) for integer string conversion: value has 5000 di ...


➡ Cần cách cộng trực tiếp trên mảng, O(n), không phụ thuộc kích thước số.

## 5. Insight — cộng tay có nhớ, đi từ phải sang trái

Nhớ lại cách cộng ở tiểu học: `274 + 181`

```text
    2 7 4
  + 1 8 1
  -------
  hàng đơn vị: 4 + 1 = 5         -> viết 5, nhớ 0
  hàng chục  : 7 + 8 + 0 = 15    -> viết 5, nhớ 1
  hàng trăm  : 2 + 1 + 1 = 4     -> viết 4, nhớ 0
  => 455
```

Mỗi hàng: `total = chữ_số_num + chữ_số_k + nhớ`; viết `total % 10`, nhớ `total // 10`.

**Mẹo sáng tạo:** thay vì tách `k` thành chữ số, hãy coi **toàn bộ `k` là số nhớ ban đầu**! Ở mỗi hàng: `k += num[i]`, viết `k % 10`, rồi `k //= 10`. Phần `k` còn lại tự động mang cả chữ số của `k` lẫn số nhớ.

In [7]:
# Thử mẹo "k là số nhớ" cho 274 + 181
num, k = [2, 7, 4], 181
for d in reversed(num):
    k += d
    print(f"cộng chữ số {d}: k = {k:>3} -> viết {k % 10}, mang sang {k // 10}")
    k //= 10
print("k còn lại sau cùng:", k)

cộng chữ số 4: k = 185 -> viết 5, mang sang 18
cộng chữ số 7: k =  25 -> viết 5, mang sang 2
cộng chữ số 2: k =   4 -> viết 4, mang sang 0
k còn lại sau cùng: 0


## 6. Lời giải chính — xây từng đoạn rồi ghép

**Đoạn 1:** duyệt `i` từ cuối mảng về đầu (`range(len(num) - 1, -1, -1)`), cộng `num[i]` vào `k`, viết `k % 10`, `k //= 10`. Kết quả viết ra là các chữ số theo thứ tự **ngược** → dùng `append` rồi đảo một lần ở cuối.

**Đoạn 2:** sau khi hết mảng, `k` có thể vẫn > 0 (do số nhớ hoặc do `k` dài hơn `num`): tiếp tục tách chữ số của `k`.

**Đoạn 3:** đảo kết quả.

In [8]:
def add_to_array_form(num: list[int], k: int) -> list[int]:
    res = []
    for i in range(len(num) - 1, -1, -1):     # Đoạn 1
        k += num[i]
        res.append(k % 10)
        k //= 10
    while k > 0:                              # Đoạn 2
        res.append(k % 10)
        k //= 10
    res.reverse()                             # Đoạn 3
    return res

print(add_to_array_form([1, 2, 0, 0], 34))
print(add_to_array_form([2, 7, 4], 181))
print(add_to_array_form([2, 1, 5], 806))
print(add_to_array_form([0], 10000))           # k dài hơn num

[1, 2, 3, 4]
[4, 5, 5]
[1, 0, 2, 1]
[1, 0, 0, 0, 0]


## 7. Trace từng bước

`num = [2, 1, 5]`, `k = 806`:

| i | num[i] | k trước | k sau khi cộng | viết (`% 10`) | k mang sang (`// 10`) | res (ngược) |
|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 2 | 5 | 806 | 811 | 1 | 81 | `[1]` |
| 1 | 1 | 81 | 82 | 2 | 8 | `[1,2]` |
| 0 | 2 | 8 | 10 | 0 | 1 | `[1,2,0]` |
| hết mảng | — | 1 | 1 | 1 | 0 | `[1,2,0,1]` |

Đảo lại → `[1, 0, 2, 1]` = 1021 ✔

In [9]:
def trace_add(num: list[int], k: int) -> list[int]:
    print(f"num = {num}, k = {k}")
    res = []
    for i in range(len(num) - 1, -1, -1):
        before = k
        k += num[i]
        res.append(k % 10)
        print(f"  i={i}: num[i]={num[i]}, k {before} + {num[i]} = {k:>4} -> viết {k % 10}, mang {k // 10}, res={res}")
        k //= 10
    while k > 0:
        res.append(k % 10)
        print(f"  hết mảng: k={k:>4} -> viết {k % 10}, mang {k // 10}, res={res}")
        k //= 10
    res.reverse()
    print("  đảo lại ->", res)
    return res

trace_add([2, 1, 5], 806)
print()
trace_add([9, 9], 1)

num = [2, 1, 5], k = 806
  i=2: num[i]=5, k 806 + 5 =  811 -> viết 1, mang 81, res=[1]
  i=1: num[i]=1, k 81 + 1 =   82 -> viết 2, mang 8, res=[1, 2]
  i=0: num[i]=2, k 8 + 2 =   10 -> viết 0, mang 1, res=[1, 2, 0]
  hết mảng: k=   1 -> viết 1, mang 0, res=[1, 2, 0, 1]
  đảo lại -> [1, 0, 2, 1]

num = [9, 9], k = 1
  i=1: num[i]=9, k 1 + 9 =   10 -> viết 0, mang 1, res=[0]
  i=0: num[i]=9, k 1 + 9 =   10 -> viết 0, mang 1, res=[0, 0]
  hết mảng: k=   1 -> viết 1, mang 0, res=[0, 0, 1]
  đảo lại -> [1, 0, 0]


[1, 0, 0]

Ví dụ thứ hai cho thấy số nhớ lan qua **mọi** chữ số và tạo thêm một chữ số mới: `99 + 1 = 100`.

## 8. Code nộp LeetCode

### Cách A — coi `k` là số nhớ (gọn nhất)

In [10]:
class Solution:
    def addToArrayForm(self, num: list[int], k: int) -> list[int]:
        res = []
        i = len(num) - 1
        while i >= 0 or k > 0:
            if i >= 0:
                k += num[i]
                i -= 1
            res.append(k % 10)
            k //= 10
        return res[::-1]

print(Solution().addToArrayForm([1, 2, 0, 0], 34))
print(Solution().addToArrayForm([2, 7, 4], 181))
print(Solution().addToArrayForm([2, 1, 5], 806))

[1, 2, 3, 4]
[4, 5, 5]
[1, 0, 2, 1]


Gộp Đoạn 1 và Đoạn 2 vào một vòng `while i >= 0 or k > 0`: còn chữ số của `num` **hoặc** còn `k` thì tiếp tục.

### Cách B — cộng chữ số với số nhớ riêng (kiểu "sách giáo khoa")

Tách `k` thành chữ số `k_digits`, rồi cộng hai mảng chữ số từ phải sang với biến `carry`. Dài hơn nhưng giống hệt cách làm trong Add Strings / Add Binary.

In [11]:
class SolutionCarry:
    def addToArrayForm(self, num: list[int], k: int) -> list[int]:
        kd = [int(c) for c in str(k)]
        i, j, carry = len(num) - 1, len(kd) - 1, 0
        res = []
        while i >= 0 or j >= 0 or carry:
            a = num[i] if i >= 0 else 0
            b = kd[j] if j >= 0 else 0
            carry, digit = divmod(a + b + carry, 10)
            res.append(digit)
            i -= 1
            j -= 1
        return res[::-1]

print(SolutionCarry().addToArrayForm([2, 1, 5], 806))
print(SolutionCarry().addToArrayForm([9, 9], 1))

[1, 0, 2, 1]
[1, 0, 0]


`a = num[i] if i >= 0 else 0` là **biểu thức điều kiện** (ternary): lấy `num[i]` nếu còn chữ số, ngược lại coi như `0`.

### Cách C — sửa tại chỗ trên `num`, chỉ chèn thêm đầu nếu cần

Duyệt từ cuối, ghi đè `num[i]`; khi hết mảng mà `k` còn, tách chữ số của `k` và **ghép vào đầu** một lần.

In [12]:
class SolutionInPlace:
    def addToArrayForm(self, num: list[int], k: int) -> list[int]:
        num = num[:]                        # bản sao để không sửa input của người gọi
        for i in range(len(num) - 1, -1, -1):
            if k == 0:
                break                       # dừng sớm: phần còn lại giữ nguyên
            k, num[i] = divmod(num[i] + k, 10)
        if k:
            return [int(c) for c in str(k)] + num
        return num

print(SolutionInPlace().addToArrayForm([1, 2, 0, 0], 34))
print(SolutionInPlace().addToArrayForm([0], 10000))

[1, 2, 3, 4]
[1, 0, 0, 0, 0]


Điểm hay: khi `k` về 0, ta **dừng sớm** — với `num` dài 10⁴ và `k` nhỏ, chỉ vài chữ số cuối bị ảnh hưởng.

### Cách D — đổi số (chỉ dùng trong Python, cần chú ý giới hạn chữ số)

In [13]:
class SolutionInt:
    def addToArrayForm(self, num: list[int], k: int) -> list[int]:
        value = 0
        for d in num:                      # tránh int(str) để không đụng giới hạn 4300 chữ số
            value = value * 10 + d
        value += k
        res = []
        while value:
            value, d = divmod(value, 10)
            res.append(d)
        return res[::-1] or [0]

print(SolutionInt().addToArrayForm([2, 7, 4], 181))

[4, 5, 5]


Cách D tránh `int(str)` nhưng mỗi phép `value * 10 + d` trên số lớn tốn O(độ dài) → tổng **O(n²)**. Chỉ nên biết để so sánh.

## 9. Vì sao đúng? (Cách A)

Gọi `V` = giá trị ban đầu của `num`, `k0` = giá trị ban đầu của `k`. Sau khi đã làm `t` vòng, đặt:
- `P` = giá trị của phần `num` **chưa** xử lý (các chữ số `num[0..i]`),
- `R` = giá trị các chữ số đã viết vào `res` (đọc ngược, `res[0]` là hàng đơn vị).

**Invariant:** `V + k0 = (P + k) · 10^t + R`.

- **Ban đầu** (`t = 0`): `P = V`, `k = k0`, `R = 0` → đúng.
- **Một vòng:** chữ số `d = num[i]` rời `P` (ta có `P = 10·P' + d`) và được cộng vào `k`. Sau đó tách `k = 10·q + r`: viết `r` vào hàng `10^t` của `R`, giữ lại `q`. Kiểm tra: `(P + k)·10^t = (10·P' + d + k)·10^t = (10·P' + 10·q + r)·10^t = (P' + q)·10^{t+1} + r·10^t` → invariant vẫn đúng với `t + 1`.
- **Kết thúc:** hết chữ số (`P = 0`) và `k = 0` → `V + k0 = R`. Đảo `res` cho đúng thứ tự trái → phải.

Mỗi chữ số viết ra nằm trong `0..9` nhờ `% 10`. Vòng lặp dừng vì mỗi vòng hoặc giảm `i`, hoặc (khi hết `num`) chia `k` cho 10.

Không có số 0 thừa ở đầu: chữ số cuối cùng được viết khi `k > 0` hoặc khi còn chữ số `num[0] ≥ 1`.

## 10. Độ phức tạp

Gọi `n = len(num)`, `m` = số chữ số của `k` (≤ 5).

| Cách | Time | Space (ngoài output) | Ghi chú |
|---|---|---|---|
| Baseline `int(str)` | siêu tuyến tính, có thể **lỗi** với > 4300 chữ số | O(n) | không nên |
| **A. k là số nhớ** | O(max(n, m)) | O(1) | gọn, chuẩn |
| B. Carry riêng | O(max(n, m)) | O(m) cho chữ số của k | khuôn tổng quát cộng 2 dãy |
| C. Tại chỗ + dừng sớm | O(n) cho bản sao; vòng cộng chỉ O(m + số lần lan nhớ) | O(n) bản sao | nhanh nhất khi được sửa input |
| D. Đổi số thủ công | O(n²) | O(n) | để so sánh |

## 11. Bộ test

In [14]:
tests = [
    ([1, 2, 0, 0], 34, [1, 2, 3, 4]),
    ([2, 7, 4], 181, [4, 5, 5]),
    ([2, 1, 5], 806, [1, 0, 2, 1]),
    ([0], 1, [1]),                         # num = 0
    ([0], 10000, [1, 0, 0, 0, 0]),         # k dài hơn num
    ([9, 9, 9], 1, [1, 0, 0, 0]),          # nhớ lan hết
    ([9], 9999, [1, 0, 0, 0, 8]),
    ([1] + [0] * 20, 5, [1] + [0] * 19 + [5]),  # num dài, k ngắn
]
impls = {
    "baseline": add_via_int, "chính": add_to_array_form,
    "LC-A": Solution().addToArrayForm, "LC-B": SolutionCarry().addToArrayForm,
    "LC-C": SolutionInPlace().addToArrayForm, "LC-D": SolutionInt().addToArrayForm,
}
for num, k, expected in tests:
    for name, f in impls.items():
        got = f(num, k)
        assert got == expected, (name, num, k, got, expected)
    print(f"{str(num)[:30]:32} + {k:<6} -> {str(expected)[:30]} PASS")
print("Tất cả test PASS!")

[1, 2, 0, 0]                     + 34     -> [1, 2, 3, 4] PASS
[2, 7, 4]                        + 181    -> [4, 5, 5] PASS
[2, 1, 5]                        + 806    -> [1, 0, 2, 1] PASS
[0]                              + 1      -> [1] PASS
[0]                              + 10000  -> [1, 0, 0, 0, 0] PASS
[9, 9, 9]                        + 1      -> [1, 0, 0, 0] PASS
[9]                              + 9999   -> [1, 0, 0, 0, 8] PASS
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0,   + 5      -> [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, PASS
Tất cả test PASS!


In [15]:
import random
rng = random.Random(989)
for _ in range(2000):
    x = rng.randint(0, 10**8)
    k = rng.randint(1, 10**4)
    num = to_digits(x)
    expected = to_digits(x + k)
    for name, f in impls.items():
        assert f(num, k) == expected, (name, num, k)
    assert num == to_digits(x), "input bị sửa!"
print("2000 test ngẫu nhiên: mọi cách khớp, input không bị sửa")

big = [9] * 10000
res = Solution().addToArrayForm(big, 1)
print("10000 chữ số 9 + 1 ->", len(res), "chữ số, bắt đầu", res[:3], "kết thúc", res[-3:])

2000 test ngẫu nhiên: mọi cách khớp, input không bị sửa
10000 chữ số 9 + 1 -> 10001 chữ số, bắt đầu [1, 0, 0] kết thúc [0, 0, 0]


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên số nhớ cuối cùng

In [16]:
def wrong_last_carry(num, k):
    res = []
    for i in range(len(num) - 1, -1, -1):
        k += num[i]
        res.append(k % 10)
        k //= 10
    return res[::-1]                 # thiếu vòng while k > 0

print("Sai :", wrong_last_carry([2, 1, 5], 806))
print("Đúng:", add_to_array_form([2, 1, 5], 806))

Sai : [0, 2, 1]
Đúng: [1, 0, 2, 1]


**Vì sao sai:** `215 + 806 = 1021` có thêm chữ số; sau khi hết `num`, `k = 1` vẫn còn. **Sửa:** thêm `while k > 0` hoặc điều kiện `while i >= 0 or k > 0`.

### Lỗi 2: Cộng từ trái sang phải

In [17]:
def wrong_left_to_right(num, k):
    res = []
    for d in num:                    # đi từ hàng cao nhất
        k += d
        res.append(k % 10)
        k //= 10
    return res

print("Sai :", wrong_left_to_right([1, 2, 0, 0], 34))
print("Đúng:", add_to_array_form([1, 2, 0, 0], 34))

Sai : [5, 5, 0, 0]
Đúng: [1, 2, 3, 4]


**Vì sao sai:** chữ số đơn vị của `k` phải cộng với chữ số **cuối** của `num`. **Sửa:** duyệt từ phải sang trái.

### Lỗi 3: Dùng `res.insert(0, d)` trong vòng lặp

Đúng kết quả nhưng mỗi lần `insert(0, …)` dời cả list → O(n²) với `n = 10⁴` (~5·10⁷ thao tác). Dùng `append` rồi đảo **một lần**.

### Lỗi 4: Dùng `/` thay vì `//`

In [18]:
k = 81
print("k / 10  =", k / 10, "(số thực, sai)")
print("k // 10 =", k // 10)

k / 10  = 8.1 (số thực, sai)
k // 10 = 8


### Lỗi 5: `return res.reverse()`

`reverse()` trả về `None` → hàm trả `None`. Viết `res.reverse()` rồi `return res`, hoặc `return res[::-1]`.

## 13. Mẹo nhận dạng pattern

- "Số được biểu diễn bằng **mảng/chuỗi/danh sách liên kết** chữ số" + phép cộng → **cộng tay có nhớ, từ hàng thấp lên**.
- Hai toán hạng **độ dài khác nhau** → vòng `while i >= 0 or j >= 0 or carry`, coi chữ số thiếu là `0`.
- Một toán hạng là **số nhỏ** → dùng mẹo "cả số là carry".
- Thêm vào cuối rồi **đảo một lần** thay vì chèn vào đầu nhiều lần.

## 14. Biến thể

1. **LeetCode 66 — Plus One:** chính là bài này với `k = 1`.
2. **LeetCode 415 — Add Strings:** cộng hai số dạng chuỗi.
3. **LeetCode 67 — Add Binary:** cộng nhị phân → thay `10` bằng `2`.
4. **LeetCode 2 — Add Two Numbers:** chữ số trong linked list, **đã** xếp ngược (hàng đơn vị trước).
5. **LeetCode 43 — Multiply Strings:** nhân hai số chuỗi bằng nhân tay.

In [19]:
def add_in_base(a: str, b: str, base: int) -> str:
    i, j, carry, out = len(a) - 1, len(b) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        s = carry
        if i >= 0:
            s += int(a[i]); i -= 1
        if j >= 0:
            s += int(b[j]); j -= 1
        carry, d = divmod(s, base)
        out.append(str(d))
    return "".join(reversed(out))

print("Add Binary  '1011' + '11'  =", add_in_base("1011", "11", 2))
print("Add Strings '456' + '77'   =", add_in_base("456", "77", 10))

Add Binary  '1011' + '11'  = 1110
Add Strings '456' + '77'   = 533


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Plus One
Cho mảng chữ số, cộng thêm 1.

In [20]:
def plus_one(digits: list[int]) -> list[int]:
    # TODO
    pass

# assert plus_one([1, 2, 9]) == [1, 3, 0]
# assert plus_one([9, 9]) == [1, 0, 0]

### Bài 2 — Cộng hai mảng chữ số
Cho hai mảng chữ số `a`, `b` (độ dài có thể khác nhau), trả về mảng chữ số của tổng.

In [21]:
def add_arrays(a: list[int], b: list[int]) -> list[int]:
    # TODO
    pass

# assert add_arrays([9, 9, 9], [1]) == [1, 0, 0, 0]
# assert add_arrays([1, 2], [3, 4, 5]) == [3, 5, 7]

### Bài 3 — Add Binary
Cộng hai chuỗi nhị phân.

In [22]:
def add_binary(a: str, b: str) -> str:
    # TODO
    pass

# assert add_binary("11", "1") == "100"
# assert add_binary("1010", "1011") == "10101"

### Bài 4 — Trừ một số nhỏ
Cho `num` (dạng mảng) và `k` với `0 <= k <= giá trị của num`, trả về dạng mảng của `num − k` (không có số 0 thừa ở đầu).

In [23]:
def subtract_from_array(num: list[int], k: int) -> list[int]:
    # TODO
    pass

# assert subtract_from_array([1, 0, 0, 0], 1) == [9, 9, 9]
# assert subtract_from_array([5, 0], 50) == [0]
# assert subtract_from_array([1, 2, 3], 23) == [1, 0, 0]

### Bài 5 — Nhân mảng chữ số với số nhỏ
Trả về dạng mảng của `num × m` với `0 <= m <= 9`.

In [24]:
def multiply_small(num: list[int], m: int) -> list[int]:
    # TODO
    pass

# assert multiply_small([1, 2, 3], 4) == [4, 9, 2]
# assert multiply_small([9, 9], 9) == [8, 9, 1]
# assert multiply_small([1, 2], 0) == [0]

### Bài 6 (nâng cao) — Multiply Strings
Nhân hai số dạng chuỗi (không đổi sang `int`).

In [25]:
def multiply_strings(a: str, b: str) -> str:
    # TODO
    pass

# assert multiply_strings("12", "34") == "408"
# assert multiply_strings("0", "999") == "0"
# assert multiply_strings("99", "99") == "9801"

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | đây là bài chính với `k = 1` | dùng lại khuôn `while i >= 0 or k` | hoặc: biến 9 cuối thành 0, tăng chữ số đầu tiên < 9 |
| 2 | hai con trỏ từ cuối | chữ số thiếu coi là 0 | `carry, d = divmod(a + b + carry, 10)` |
| 3 | như bài 2 | cơ số 2 | `divmod(s, 2)` rồi nối chuỗi ngược |
| 4 | trừ có **mượn** (borrow) | `d = num[i] − (k % 10) − borrow`; nếu `d < 0` thì `+10`, mượn 1 | sau cùng bỏ số 0 đầu (giữ ít nhất một chữ số) |
| 5 | coi `carry` như bài chính | `carry, d = divmod(num[i]*m + carry, 10)` | kết quả 0 → `[0]` |
| 6 | nhân tay: `a[i]*b[j]` rơi vào ô `i + j + 1` | mảng kết quả độ dài `len(a)+len(b)` | xử lý nhớ, bỏ số 0 đầu |

## 17. Đáp án tham khảo

In [26]:
def ans_plus_one(digits):
    return Solution().addToArrayForm(digits, 1)

def ans_add_arrays(a, b):
    i, j, carry, res = len(a) - 1, len(b) - 1, 0, []
    while i >= 0 or j >= 0 or carry:
        s = carry + (a[i] if i >= 0 else 0) + (b[j] if j >= 0 else 0)
        carry, d = divmod(s, 10)
        res.append(d)
        i -= 1; j -= 1
    return res[::-1]

def ans_add_binary(a, b):
    return add_in_base(a, b, 2)

def ans_subtract_from_array(num, k):
    res = num[:]
    borrow = 0
    i = len(res) - 1
    while i >= 0 and (k > 0 or borrow):
        d = res[i] - k % 10 - borrow
        k //= 10
        if d < 0:
            d += 10
            borrow = 1
        else:
            borrow = 0
        res[i] = d
        i -= 1
    while len(res) > 1 and res[0] == 0:
        res.pop(0)
    return res

def ans_multiply_small(num, m):
    if m == 0:
        return [0]
    carry, res = 0, []
    for d in reversed(num):
        carry, digit = divmod(d * m + carry, 10)
        res.append(digit)
    while carry:
        carry, digit = divmod(carry, 10)
        res.append(digit)
    return res[::-1]

def ans_multiply_strings(a, b):
    out = [0] * (len(a) + len(b))
    for i in range(len(a) - 1, -1, -1):
        for j in range(len(b) - 1, -1, -1):
            s = out[i + j + 1] + int(a[i]) * int(b[j])
            out[i + j + 1] = s % 10
            out[i + j] += s // 10
    text = "".join(map(str, out)).lstrip("0")
    return text or "0"

assert ans_plus_one([1, 2, 9]) == [1, 3, 0]
assert ans_plus_one([9, 9]) == [1, 0, 0]
assert ans_add_arrays([9, 9, 9], [1]) == [1, 0, 0, 0]
assert ans_add_arrays([1, 2], [3, 4, 5]) == [3, 5, 7]
assert ans_add_binary("11", "1") == "100"
assert ans_add_binary("1010", "1011") == "10101"
assert ans_subtract_from_array([1, 0, 0, 0], 1) == [9, 9, 9]
assert ans_subtract_from_array([5, 0], 50) == [0]
assert ans_subtract_from_array([1, 2, 3], 23) == [1, 0, 0]
assert ans_multiply_small([1, 2, 3], 4) == [4, 9, 2]
assert ans_multiply_small([9, 9], 9) == [8, 9, 1]
assert ans_multiply_small([1, 2], 0) == [0]
assert ans_multiply_strings("12", "34") == "408"
assert ans_multiply_strings("0", "999") == "0"
assert ans_multiply_strings("99", "99") == "9801"
for _ in range(500):
    x, k = rng.randint(0, 10**6), rng.randint(0, 10**6)
    if k <= x:
        assert ans_subtract_from_array(to_digits(x), k) == to_digits(x - k)
    assert ans_multiply_strings(str(x), str(k)) == str(x * k)
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def addToArrayForm(self, num: list[int], k: int) -> list[int]:
        res, i = [], len(num) - 1
        while i >= 0 or k > 0:
            if i >= 0:
                k += num[i]
                i -= 1
            res.append(k % 10)
            k //= 10
        return res[::-1]
```

Khuôn cộng tổng quát: `while i >= 0 or j >= 0 or carry:` → `carry, d = divmod(a + b + carry, base)` → `append(d)` → đảo một lần.

**Checklist trước khi submit**
- [ ] Duyệt từ **phải sang trái** (hàng đơn vị trước)?
- [ ] Xử lý số nhớ / phần `k` còn lại sau khi hết `num`?
- [ ] Dùng `//` (không phải `/`)?
- [ ] `append` + đảo một lần, không `insert(0, …)` trong vòng lặp?
- [ ] Không `return res.reverse()`?
- [ ] Thử `[9,9,9] + 1` và `[0] + 10000`?